# Week 13 Integrative Assignment: TrustFirst's Risk Revolution

**ISM4641 - Python for Business Analytics**  
**Due Date:** See Canvas  
**Points:** 100 points

---

## The Story

**Monday, 8:30 AM**

James Washington takes a deep breath as he enters the executive conference room at **TrustFirst Credit Union**. As a Credit Analyst in the Risk Management department, he's been called to present a solution to a problem that's been keeping leadership up at night.

The Chief Risk Officer, Patricia Okonkwo, opens the meeting.

*"Last quarter, our loan default rate increased to 4.2%—the highest in five years. That cost us $1.8 million in write-offs. Our current loan approval process relies heavily on credit scores and manual review. It's inconsistent, and we're either being too aggressive—approving risky loans—or too conservative—turning away good members."*

*"James, you've been working with our historical loan data. The board wants a data-driven approach. Can you build a model that predicts which loan applications are likely to default? We need to make smarter decisions—protect the credit union while still serving our members."*

James has been preparing for this. He has three years of loan performance data: 2,500 loans with complete records showing which ones defaulted and which performed well. Time to build a model that could save TrustFirst millions.

*"I can have a recommendation by Friday,"* James says confidently.

---

## Your Mission

You are James. Build a logistic regression model to predict loan defaults and develop a risk-based decision framework for TrustFirst Credit Union.

**Skills you'll apply:**
- Logistic regression for binary classification
- Train/test splitting
- Confusion matrix interpretation
- Precision, recall, and accuracy evaluation
- Probability predictions for risk scoring
- Coefficient interpretation (odds ratios)
- Business cost-benefit analysis

---

## Assignment Requirements

1. Complete all sections of this notebook
2. Build and evaluate classification models
3. Interpret results in business terms
4. Include your video walkthrough (3-5 minutes)

**Video Walkthrough:** Present your risk model as if presenting to the Credit Union's board of directors.

---

## Student Information

**Name:** [Enter your name here]  
**USF ID:** [Enter your ID here]  
**Date:** [Enter date here]

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set random seed for reproducibility
np.random.seed(1341)

# Generate loan application data
n_loans = 2500

# Applicant characteristics
annual_income = np.random.lognormal(10.8, 0.5, n_loans)  # Log-normal distribution
annual_income = np.clip(annual_income, 25000, 250000).round(-2)

# Credit score (300-850)
credit_score = np.random.normal(680, 80, n_loans)
credit_score = np.clip(credit_score, 350, 850).round(0)

# Debt-to-income ratio
dti_ratio = np.random.beta(2, 5, n_loans) * 60  # 0-60%
dti_ratio = dti_ratio.round(1)

# Employment tenure (years at current job)
employment_years = np.random.exponential(4, n_loans)
employment_years = np.clip(employment_years, 0, 30).round(1)

# Loan amount requested
loan_amount = np.random.lognormal(9.5, 0.6, n_loans)
loan_amount = np.clip(loan_amount, 5000, 100000).round(-2)

# Loan purpose
loan_purpose = np.random.choice(
    ['Auto', 'Home Improvement', 'Debt Consolidation', 'Personal', 'Medical'],
    n_loans, p=[0.30, 0.20, 0.25, 0.15, 0.10]
)

# Previous defaults (0 or 1)
prior_defaults = np.random.choice([0, 1], n_loans, p=[0.88, 0.12])

# Calculate default probability based on features
# Higher income, credit score, employment = lower risk
# Higher DTI, loan amount, prior defaults = higher risk
risk_score = (
    -5.5 +                                          # Base
    -0.00002 * annual_income +                     # Income reduces risk
    -0.015 * credit_score +                        # Better credit = less risk
    0.08 * dti_ratio +                             # Higher DTI = more risk
    -0.05 * employment_years +                     # Stable employment = less risk
    0.00003 * loan_amount +                        # Larger loans = more risk
    1.5 * prior_defaults +                         # Prior defaults = much more risk
    np.random.normal(0, 0.5, n_loans)              # Random variation
)

# Convert to probability using sigmoid function
default_prob = 1 / (1 + np.exp(-risk_score))

# Generate actual defaults based on probability
defaulted = (np.random.random(n_loans) < default_prob).astype(int)

# Create DataFrame
loans = pd.DataFrame({
    'loan_id': range(1, n_loans + 1),
    'annual_income': annual_income,
    'credit_score': credit_score,
    'dti_ratio': dti_ratio,
    'employment_years': employment_years,
    'loan_amount': loan_amount,
    'loan_purpose': loan_purpose,
    'prior_defaults': prior_defaults,
    'defaulted': defaulted
})

# New loan applications to evaluate
new_applications = pd.DataFrame({
    'applicant_id': ['APP001', 'APP002', 'APP003', 'APP004', 'APP005', 'APP006'],
    'name': ['Maria Santos', 'Robert Chen', 'Lisa Johnson', 'Marcus Williams', 'Sarah Miller', 'David Kim'],
    'annual_income': [65000, 95000, 42000, 78000, 55000, 120000],
    'credit_score': [720, 680, 590, 710, 640, 750],
    'dti_ratio': [25.0, 35.0, 45.0, 28.0, 38.0, 20.0],
    'employment_years': [5.5, 2.0, 1.5, 8.0, 3.0, 12.0],
    'loan_amount': [15000, 35000, 8000, 25000, 18000, 50000],
    'loan_purpose': ['Auto', 'Home Improvement', 'Medical', 'Debt Consolidation', 'Personal', 'Home Improvement'],
    'prior_defaults': [0, 0, 1, 0, 0, 0]
})

print("TrustFirst Credit Union Loan Data Loaded Successfully!")
print(f"\n  Historical Loans: {len(loans)}")
print(f"  Default Rate: {loans['defaulted'].mean()*100:.1f}%")
print(f"  New Applications to Evaluate: {len(new_applications)}")

print("\n" + "="*70)
print("HISTORICAL LOAN DATA PREVIEW:")
display(loans.head(10))

print("\nNEW LOAN APPLICATIONS:")
display(new_applications)

---

## Part 1: Understanding the Data (15 points)

Before building a model, James needs to understand what factors are associated with defaults in TrustFirst's historical data.

**Your Task:**
1. Calculate the overall default rate in the historical data
2. Compare the average characteristics of defaulted vs non-defaulted loans:
   - Mean income, credit score, DTI ratio, employment years, loan amount
3. Calculate the default rate by loan purpose
4. Calculate the default rate by prior default history
5. Write a brief summary of what factors seem most associated with default risk

In [ ]:
# Part 1: Understanding the Data

# 1. Overall default rate
print("LOAN PORTFOLIO ANALYSIS")
print("="*60)


# 2. Compare defaulted vs non-defaulted loan characteristics
print("\nCOMPARISON: DEFAULTED vs NON-DEFAULTED LOANS")
print("-"*60)


# 3. Default rate by loan purpose
print("\nDEFAULT RATE BY LOAN PURPOSE")
print("-"*60)


# 4. Default rate by prior default history
print("\nDEFAULT RATE BY PRIOR DEFAULT HISTORY")
print("-"*60)


# 5. Summary of risk factors
print("\nKEY RISK FACTORS IDENTIFIED")
print("-"*60)


---

## Part 2: Building the Default Prediction Model (25 points)

Now James needs to build a logistic regression model that can predict which loans are likely to default.

**Your Task:**
1. Prepare the features (use: annual_income, credit_score, dti_ratio, employment_years, loan_amount, prior_defaults)
2. Split the data into training (80%) and testing (20%) sets with random_state=42
3. Train a LogisticRegression model
4. Make predictions on the test set
5. Calculate and report the accuracy
6. Display and interpret the confusion matrix:
   - What does each cell mean in this business context?
   - What are True Positives, False Positives, True Negatives, False Negatives in terms of loans?

In [ ]:
# Part 2: Building the Default Prediction Model

# 1. Prepare features and target
feature_names = ['annual_income', 'credit_score', 'dti_ratio', 
                 'employment_years', 'loan_amount', 'prior_defaults']


# 2. Split data (80/20)


# 3. Train logistic regression model


# 4. Make predictions


# 5. Calculate accuracy
print("DEFAULT PREDICTION MODEL")
print("="*60)


# 6. Confusion matrix with interpretation
print("\nCONFUSION MATRIX")
print("-"*60)


print("\nINTERPRETATION IN BUSINESS TERMS")
print("-"*60)
# Explain what each cell means:
# True Negatives (TN): 
# False Positives (FP): 
# False Negatives (FN): 
# True Positives (TP): 


---

## Part 3: Model Performance Analysis (20 points)

Patricia asks: *"Accuracy isn't enough. I need to understand the trade-offs. When we're wrong, what kind of wrong are we?"*

**Your Task:**
1. Generate and display the classification report
2. Extract and explain precision for predicting defaults:
   - What does this mean in business terms?
3. Extract and explain recall for predicting defaults:
   - What does this mean in business terms?
4. Calculate the cost of errors:
   - False Negative (missed default): Average loan amount × 60% (assume 60% loss on default)
   - False Positive (rejected good loan): Average loan amount × 10% (lost interest income opportunity)
5. Based on these costs, is precision or recall more important? Justify your answer.

In [ ]:
# Part 3: Model Performance Analysis

# 1. Classification report
print("CLASSIFICATION REPORT")
print("="*60)


# 2. Precision interpretation
print("\nPRECISION ANALYSIS")
print("-"*60)


# 3. Recall interpretation
print("\nRECALL ANALYSIS")
print("-"*60)


# 4. Cost of errors calculation
print("\nCOST OF ERRORS ANALYSIS")
print("-"*60)

# Calculate average loan amount


# False Negative cost (approving a loan that defaults)


# False Positive cost (rejecting a good loan)


# 5. Precision vs Recall recommendation
print("\nSTRATEGIC RECOMMENDATION: PRECISION vs RECALL")
print("-"*60)


---

## Part 4: Risk Factor Analysis (20 points)

The board wants to understand: *"What actually causes someone to default? What should our loan officers look for?"*

**Your Task:**
1. Extract and display the model coefficients for each feature
2. Calculate the odds ratio for each feature: exp(coefficient)
3. Create a horizontal bar chart showing the coefficients (positive vs negative)
4. Rank features by their impact on default risk
5. Interpret the top 3 most influential features in business terms:
   - For each, explain what a one-unit increase means for default odds
6. Provide actionable insights for loan officers based on these findings

In [ ]:
# Part 4: Risk Factor Analysis

# 1. Extract coefficients
print("MODEL COEFFICIENTS")
print("="*60)


# 2. Calculate odds ratios
print("\nODDS RATIOS")
print("-"*60)


# 3. Bar chart of coefficients


# 4. Rank features by impact
print("\nFEATURE IMPORTANCE RANKING")
print("-"*60)


# 5. Interpret top 3 features
print("\nTOP RISK FACTORS INTERPRETATION")
print("-"*60)


# 6. Actionable insights for loan officers
print("\nGUIDANCE FOR LOAN OFFICERS")
print("-"*60)


---

## Part 5: Evaluating New Applications (20 points)

It's Friday. Time to put the model to work. Six loan applications are waiting for decisions.

**Your Task:**
1. Use predict_proba() to get default probability for each new application
2. Create a risk classification system:
   - Low Risk: <20% default probability → Auto-approve
   - Medium Risk: 20-50% → Manual review required
   - High Risk: >50% → Auto-decline
3. Classify each applicant and create a summary table
4. For applicants requiring manual review, identify which factors are most concerning
5. Create the final board recommendation including:
   - Model performance summary
   - Expected savings from using the model
   - Decisions on the 6 pending applications
   - Recommendations for implementation

In [ ]:
# Part 5: Evaluating New Applications

# Prepare new application features
X_new = new_applications[feature_names]

# 1. Get default probabilities


# 2-3. Risk classification and summary table
print("NEW APPLICATION RISK ASSESSMENT")
print("="*70)

# Create classification function


# Display results


# 4. Manual review details
print("\nMANUAL REVIEW REQUIRED - DETAILED ANALYSIS")
print("-"*70)


# 5. Board Recommendation
print("\n" + "="*70)
print("              TRUSTFIRST CREDIT UNION")
print("        CREDIT RISK MODEL - BOARD RECOMMENDATION")
print("              Prepared by: [Your Name]")
print("="*70)

# Model Performance Summary
print("\n1. MODEL PERFORMANCE SUMMARY")
print("-"*70)


# Expected Savings
print("\n2. EXPECTED FINANCIAL IMPACT")
print("-"*70)
# Calculate expected reduction in defaults using the model


# Application Decisions
print("\n3. PENDING APPLICATION DECISIONS")
print("-"*70)


# Implementation Recommendations
print("\n4. IMPLEMENTATION RECOMMENDATIONS")
print("-"*70)


# Model Limitations
print("\n5. MODEL LIMITATIONS & CAVEATS")
print("-"*70)


print("\n" + "="*70)
print("                    END OF REPORT")
print("="*70)

---

## Epilogue

**The Following Monday**

James is reviewing emails when Patricia Okonkwo stops by his desk.

*"The board approved the risk model for pilot implementation. We're starting with auto-approvals for low-risk applications—that alone should cut our processing time by 40%."*

*"They were particularly impressed by the cost analysis. Showing the dollar impact of false positives versus false negatives made the abstract concepts concrete. The CFO said it was the clearest risk presentation she's seen."*

*"We're also going to implement your loan officer guidance. The factors you identified—prior defaults, credit score, DTI ratio—will be highlighted in our review interface."*

She pauses at the door. *"One more thing. We're creating a new Analytics Lead position in Risk Management. Your name came up. Interested?"*

James smiles. The model was just the beginning. *"Very interested. I have some ideas about expanding this to other credit products."*

---

## Interaction Log

### AI Tool Usage
*Did you use any AI tools? If so, what for?*

[Your response here]

### Classification Concepts Applied
*Explain how you used confusion matrix, precision, and recall to evaluate the model.*

[Your response here]

### Business Decision Framework
*How did you translate model probabilities into actionable lending decisions?*

[Your response here]

---

## Submission Checklist

- [ ] All code cells run without errors
- [ ] All five parts are completed
- [ ] Logistic regression model built and evaluated
- [ ] Confusion matrix displayed and interpreted
- [ ] Precision and recall explained in business terms
- [ ] Cost analysis completed for false positives and false negatives
- [ ] Coefficients interpreted as odds ratios
- [ ] Risk classification system implemented
- [ ] Board recommendation is complete and actionable
- [ ] Interaction log is completed
- [ ] Video walkthrough is recorded (3-5 minutes)

---
*ISM4641 Python for Business Analytics - Week 13 Integrative Assignment*